### Setup

In [3]:
# Library
import os
import torch
import json
from metric import * 
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore') 

# GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device : ',device)   

# CONFIG
TEST_SIZE = 100

# PATH
CONFIG_PATH = '../config.json'
with open(CONFIG_PATH,'r') as f:
    config = json.load(f)
## DATA
SEED_IMAGE_FOLDER = config.get('SEED_IMAGE_FOLDER')
SEED_LABEL_FOLDER = config.get('SEED_LABEL_FOLDER')
SEED_LABEL_FILE = sorted(os.listdir(SEED_LABEL_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
AUGEMNT_IMAGE_FOLDER = config.get('AUGEMNT_IMAGE_FOLDER')
AUGEMNT_LABEL_FOLDER =  config.get('AUGEMNT_LABEL_FOLDER')
AUGMENT_LABEL_FILE = sorted(os.listdir(AUGEMNT_LABEL_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
PROMPT_FOLDER = config.get('PROMPT_FOLDER')
PROMPT_FILE = sorted(os.listdir(PROMPT_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
ANSWER_IMAGE_FOLDER = config.get('ANSWER_IMAGE_FOLDER')
ANSWER_IMAGE_FILE = sorted(os.listdir(ANSWER_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
SD_IMAGE_FOLDER = config.get('SD_IMAGE_FOLDER')
SD_IMAGE_FILE = sorted(os.listdir(SD_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
FIGMA_IMAGE_FOLDER = config.get('FIGMA_IMAGE_FOLDER')
FIGMA_IMAGE_FILE = sorted(os.listdir(FIGMA_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
## MODEL
PRE_TRAINED_MODEL_NAME="stablediffusionapi/deliberate-v2"
SAVE_WEIGHTS_PATH = '../Experiment/model_weights/FIGMA_weights_20250226_130326'

device :  cuda


### Augment Instruction Performace

- seed image - seed label 

In [ ]:
image_folder = SEED_IMAGE_FOLDER
prompt_folder = SEED_LABEL_FOLDER
prompt_file = SEED_LABEL_FILE

clip_scores = []
for filename in tqdm(prompt_file):
    if filename.endswith('.json'):
        prompt_path = os.path.join(prompt_folder, filename)
        image_path = os.path.join(image_folder, filename.replace('.json', '.jpg'))

    with open(prompt_path, 'r') as f:
        prompt_data = json.load(f)
        persona = prompt_data.get("Prompt", {}).get("persona", "")
        task_description = prompt_data.get("Prompt", {}).get("task_description", "")
        constraint = prompt_data.get("Prompt", {}).get("constraint", "")
        caption = prompt_data.get("Input", {}).get("caption", "")
        add_info = prompt_data.get("Add_Info", "")
        
        final_prompt = " ".join([persona, task_description, constraint, caption, add_info])
        clip_score = calculate_clip_score(image_path, final_prompt)
        clip_scores.append(clip_score)
        
average_clip_score = sum(clip_scores) / len(clip_scores)
print(f"Average CLIP Score: {average_clip_score:.4f}")

- augment image - augment label

In [ ]:
image_folder = AUGEMNT_IMAGE_FOLDER
prompt_folder = AUGEMNT_LABEL_FOLDER
prompt_file = AUGMENT_LABEL_FILE

clip_scores = []
for filename in tqdm(prompt_file):
    if filename.endswith('.json'):
        prompt_path = os.path.join(prompt_folder, filename)
        image_path = os.path.join(image_folder, filename.replace('.json', '.jpg'))

    with open(prompt_path, 'r') as f:
        prompt_data = json.load(f)
        prompt = prompt_data.get("Prompt", "")
        caption = prompt_data.get("Input", {}).get("caption", "")
        add_info = prompt_data.get("Add_Info", "")
        
        final_prompt = " ".join([prompt, caption, add_info])
        clip_score = calculate_clip_score(image_path, final_prompt)
        clip_scores.append(clip_score)
        
average_clip_score = sum(clip_scores) / len(clip_scores)
print(f"Average CLIP Score: {average_clip_score:.4f}")

### Generate Model Performance-Prompt Following Performance

- stablediffusion image - prompt

In [ ]:
image_folder = SD_IMAGE_FOLDER
prompt_folder = PROMPT_FOLDER
prompt_file = PROMPT_FILE

clip_scores = []
for filename in tqdm(prompt_file):
    if filename.endswith('.json'):
        prompt_path = os.path.join(prompt_folder, filename)
        image_path = os.path.join(image_folder, filename.replace('.json', '.jpg'))

    with open(prompt_path, 'r') as f:
        prompt_data = json.load(f)
        final_prompt = prompt_data['summary']
        clip_score = calculate_clip_score(image_path, final_prompt)
        clip_scores.append(clip_score)
        
average_clip_score = sum(clip_scores) / len(clip_scores)
print(f"Average CLIP Score: {average_clip_score:.4f}")

- figma image - prompt

In [ ]:
image_folder = FIGMA_IMAGE_FOLDER
prompt_folder = PROMPT_FOLDER
prompt_file = PROMPT_FILE

clip_scores = []
for filename in tqdm(prompt_file):
    if filename.endswith('.json'):
        prompt_path = os.path.join(prompt_folder, filename)
        image_path = os.path.join(image_folder, filename.replace('.json', '.jpg'))

    with open(prompt_path, 'r') as f:
        prompt_data = json.load(f)
        final_prompt = prompt_data['summary']
        clip_score = calculate_clip_score(image_path, final_prompt)
        clip_scores.append(clip_score)
        
average_clip_score = sum(clip_scores) / len(clip_scores)
print(f"Average CLIP Score: {average_clip_score:.4f}")

### Generate Model Performance-Image Following Performance

- stablediffusion image - answer image

In [ ]:
fid_score_sd = calculate_fid(ANSWER_IMAGE_FOLDER,ANSWER_IMAGE_FILE,SD_IMAGE_FOLDER,SD_IMAGE_FILE)
lpips_sd = average_lpips(ANSWER_IMAGE_FOLDER, SD_IMAGE_FOLDER)
clip_similarity_sd = average_clip_similarity(ANSWER_IMAGE_FOLDER, SD_IMAGE_FOLDER)

print(f"SD's Average FID Score: {fid_score_sd:.4f}")
print(f"SD's Average LPIPS Score: {lpips_sd:.4f}")
print(f"SD's Average CLIP Similarity: {clip_similarity_sd:.4f}")

- figma image - answer image

In [ ]:
fid_score_figma = calculate_fid(ANSWER_IMAGE_FOLDER,ANSWER_IMAGE_FILE,FIGMA_IMAGE_FOLDER,FIGMA_IMAGE_FILE)
lpips_figma = average_lpips(ANSWER_IMAGE_FOLDER, FIGMA_IMAGE_FOLDER)
clip_similarity_figma = average_clip_similarity(ANSWER_IMAGE_FOLDER, SD_IMAGE_FOLDER)

print(f"FIGMA's Average FID Score: {fid_score_figma:.4f}")
print(f"FIGMA's Average LPIPS Score: {lpips_figma:.4f}")
print(f"FIGMA's Average CLIP Similarity: {clip_similarity_figma:.4f}")